# 01 — Corpus EDA

Stats over the FTSE 100 annual-report corpus. Run after `python -m src.ingestion.scraper` has downloaded PDFs into `data/raw/`.

Outputs `data/processed/corpus_stats.csv` for use in the README hero metrics.

In [ ]:
# Path setup: chdir to project root + add to sys.path so imports + .env paths both work.
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside filings-rag/notebooks/.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')


In [ ]:
from pathlib import Path

import csv
import fitz  # PyMuPDF
import pandas as pd

In [ ]:
RAW = Path("data/raw")
MANIFEST = Path("data/filings_manifest.csv")

manifest = pd.read_csv(MANIFEST)
print(f"Manifest rows: {len(manifest)}")
print(f"URLs populated: {manifest['pdf_url'].notna().sum()}")
print(f"By fiscal year: {manifest.groupby('fiscal_year').size().to_dict()}")

In [ ]:
# Discover downloaded PDFs
pdfs = sorted(RAW.rglob("*.pdf"))
print(f"PDFs on disk: {len(pdfs)}")
for p in pdfs[:5]:
    print(" ", p.relative_to(RAW))

In [ ]:
# Per-PDF stats: page count + file size
records = []
for p in pdfs:
    ticker = p.parent.name
    year = int(p.stem)
    size_mb = p.stat().st_size / (1024 * 1024)
    try:
        with fitz.open(p) as doc:
            page_count = doc.page_count
    except Exception as e:
        page_count = -1
        print(f"  parse failed: {p.name} — {e}")
    records.append({"ticker": ticker, "year": year, "pages": page_count, "size_mb": round(size_mb, 1)})

stats = pd.DataFrame(records)
stats

In [ ]:
# Distribution summary
if len(stats):
    print("Pages:", stats["pages"].describe().to_dict())
    print("Size MB:", stats["size_mb"].describe().to_dict())

In [ ]:
# Sample first 500 chars of 3 random PDFs (sanity check for parse quality)
import random
if pdfs:
    for p in random.sample(pdfs, min(3, len(pdfs))):
        with fitz.open(p) as doc:
            text = (doc[0].get_text() or "")[:500]
        print(f"--- {p.relative_to(RAW)} ---")
        print(text)
        print()

In [ ]:
# Persist for the README hero metrics
out = Path("data/processed")
out.mkdir(parents=True, exist_ok=True)
stats.to_csv(out / "corpus_stats.csv", index=False)
print(f"Wrote {out / 'corpus_stats.csv'}")